# 02 - Clean, data quality checks, outliers, reconciliation

Everything is code, so anyone can re-run it from the raw file. Nothing is fixed by hand in Excel.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
FUND_FILE = "../data/raw/fund_level_back_series_2004_2025.xlsx"
BULLETIN_FILE = "../data/raw/bulletin_entities_2015_2025.xlsx"

## 1. Load Table 2a and 2b and stack them

In [2]:
def load_table(sheet):
    t = pd.read_excel(FUND_FILE, sheet_name=sheet, header=4).iloc[2:]   # drop form-item and units rows
    t["balance_date_group"] = "30 June" if sheet == "Table 2a" else "Other"
    return t

df_raw = pd.concat([load_table("Table 2a"), load_table("Table 2b")], ignore_index=True)
print(df_raw.shape)

(1943, 29)


## 2. Tidy column names
Lower case, underscores, no symbols. Then rename the long ones to short, clear names.

In [3]:
df = df_raw.copy()
df.columns = (df.columns.astype(str).str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

df = df.rename(columns={
    "fund_s_rse_licensee": "licensee",
    "rse_regulatory_classification": "regulatory_class",
    "rse_membership_base": "membership_base",
    "rse_balance_date": "balance_date",
    "duration": "duration_months",
    "rse_wind_up_date": "wind_up_date",
    "total_assets": "total_assets_k",
    "proportion_of_benefits_which_are_defined_benefits": "defined_benefit_share",
    "number_of_investment_options": "investment_options",
    "number_of_mysuper_products_authorised": "mysuper_products",
    "proportion_of_total_assets_in_default_or_mysuper_strategy": "default_strategy_share",
    "net_members_benefits_outflow_ratio": "net_outflow_ratio",
    "one_year_rate_of_return": "return_1y",
    "five_year_rate_of_return": "return_5y",
    "ten_year_rate_of_return": "return_10y",
    "total_number_of_member_accounts": "member_accounts",
    "total_members_benefits": "members_benefits_k",
    "average_member_account_balance": "avg_balance_k",
    "growth_in_number_of_member_accounts": "member_account_growth",
})
list(df.columns)

['period',
 'fund_name',
 'abn',
 'regulatory_class',
 'fund_type',
 'membership_base',
 'licensee',
 'rse_licensee_ownership_type',
 'rse_licensee_profit_status',
 'rse_licensee_board_structure',
 'balance_date',
 'duration_months',
 'wind_up_date',
 'total_assets_k',
 'defined_benefit_share',
 'investment_options',
 'mysuper_products',
 'default_strategy_share',
 'net_outflow_ratio',
 'investment_expenses_ratio',
 'operating_expense_ratio',
 'return_1y',
 'return_5y',
 'return_10y',
 'member_accounts',
 'members_benefits_k',
 'avg_balance_k',
 'member_account_growth',
 'balance_date_group']

## 3. Remove footnote rows and empty rows
A real row has a date in `period`. Anything else is a note.

In [4]:
before = len(df)
df["period"] = pd.to_datetime(df["period"], errors="coerce")
df = df[df["period"].notna()].dropna(how="all").copy()
df["year"] = df["period"].dt.year
print(f"Removed {before - len(df)} non-data rows; {len(df)} fund-year rows left")

Removed 4 non-data rows; 1939 fund-year rows left


## 4. Record privacy masks BEFORE converting to numbers
If we convert first, `*` and blank both become NaN and we lose the difference. Checking first keeps it.

In [5]:
num_cols = ["total_assets_k", "member_accounts", "members_benefits_k", "avg_balance_k",
            "return_1y", "return_5y", "return_10y", "operating_expense_ratio",
            "investment_expenses_ratio", "net_outflow_ratio", "duration_months"]

masked = df[num_cols].astype(str).apply(lambda s: s.str.strip().eq("*"))
df["masked_field_count"] = masked.sum(axis=1)

for c in num_cols:
    df[c] = pd.to_numeric(df[c].replace(["*", "n/a", "np", "-", ""], np.nan), errors="coerce")

df["fund_name"] = df["fund_name"].astype(str).str.strip()
df["abn"] = df["abn"].astype("Int64").astype(str)
df["fund_type_group"] = df["fund_type"].replace({"Retail - ERF": "Retail"})
masked.sum().sort_values(ascending=False).to_frame("masked_rows")

,masked_rows
total_assets_k,248
net_outflow_ratio,244
return_1y,234
operating_expense_ratio,234
investment_expenses_ratio,234
return_5y,230
return_10y,221
members_benefits_k,205
member_accounts,205
avg_balance_k,204


## 5. Data quality checks
Each check is one True/False column. That makes it easy to count, filter, and show per row which rules failed.

| Check | Rule | Why it matters |
|---|---|---|
| missing_assets | total assets is blank or masked | can't use the fund in totals |
| any_masked_field | at least one value is `*` | privacy mask, expected for small funds |
| negative_assets | total assets < 0 | impossible |
| dup_abn_period | same ABN + period appears twice | double counting in totals |
| members_but_no_assets | accounts > 0 but assets 0 or blank | broken record |
| return_out_of_range | 1-yr return outside -50% to +50% | likely data entry or small-fund distortion |
| expense_ratio_high | operating expenses > 5% of assets | implausible for an active fund |
| avg_balance_mismatch | benefits / accounts differs from APRA's average balance by > 5% | fields disagree with each other |
| non_12_month_duration | the return covers less than 12 months | returns not comparable |
| missing_fund_type | fund type blank | can't group or reconcile |

In [6]:
checks = pd.DataFrame(index=df.index)
checks["missing_assets"]        = df["total_assets_k"].isna()
checks["any_masked_field"]      = df["masked_field_count"] > 0
checks["negative_assets"]       = df["total_assets_k"] < 0
checks["dup_abn_period"]        = df.duplicated(subset=["abn", "period"], keep=False)
checks["members_but_no_assets"] = (df["member_accounts"] > 0) & (df["total_assets_k"].fillna(0) == 0)
checks["return_out_of_range"]   = df["return_1y"].notna() & ~df["return_1y"].between(-0.5, 0.5)
checks["expense_ratio_high"]    = df["operating_expense_ratio"] > 0.05

calc_avg = np.where(df["member_accounts"] > 0, df["members_benefits_k"] / df["member_accounts"], np.nan)
checks["avg_balance_mismatch"]  = (np.abs(calc_avg - df["avg_balance_k"]) / df["avg_balance_k"]) > 0.05
checks["non_12_month_duration"] = df["duration_months"].notna() & (df["duration_months"] != 12)
checks["missing_fund_type"]     = df["fund_type"].isna()

df["dq_issue_count"] = checks.sum(axis=1)
df["dq_flag"] = df["dq_issue_count"] > 0

summary = checks.sum().sort_values(ascending=False).to_frame("rows_failed")
summary["pct_of_rows"] = (summary["rows_failed"] / len(df) * 100).round(1)
summary

,rows_failed,pct_of_rows
any_masked_field,249,12.8
missing_assets,248,12.8
avg_balance_mismatch,33,1.7
expense_ratio_high,23,1.2
return_out_of_range,13,0.7
dup_abn_period,8,0.4
non_12_month_duration,5,0.3
negative_assets,0,0.0
members_but_no_assets,0,0.0
missing_fund_type,0,0.0


In [7]:
print(f"Rows with at least one issue: {df['dq_flag'].sum()} of {len(df)} ({df['dq_flag'].mean():.1%})")

Rows with at least one issue: 307 of 1939 (15.8%)


## 6. Look at examples for every failing check
A count alone doesn't tell me much. For each check I looked at real rows to decide if it's a real problem or expected.

In [8]:
show = ["year", "fund_name", "fund_type", "total_assets_k", "member_accounts", "return_1y",
        "operating_expense_ratio", "duration_months", "wind_up_date"]
for name in summary.index[summary["rows_failed"] > 0]:
    print(f"\n=== {name} ({summary.loc[name, 'rows_failed']} rows) ===")
    print(df.loc[checks[name], show].head(3).to_string())


=== any_masked_field (249 rows) ===
     year                                                        fund_name fund_type  total_assets_k  member_accounts  return_1y  operating_expense_ratio  duration_months wind_up_date
36   2025          Macquarie University Professorial Superannuation Scheme    Retail             NaN              NaN        NaN                      NaN               12          NaT
66   2025  The University of Wollongong Professorial Superannuation Scheme    Retail             NaN              NaN        NaN                      NaN               12          NaT
115  2024          Macquarie University Professorial Superannuation Scheme    Retail             NaN              NaN        NaN                      NaN               12          NaT

=== missing_assets (248 rows) ===
     year                                                        fund_name fund_type  total_assets_k  member_accounts  return_1y  operating_expense_ratio  duration_months wind_up_date
36   202

In [9]:
# Are flagged funds small? Compare median assets of flagged vs clean rows
df.groupby("dq_flag")["total_assets_k"].median().div(1000).round(1).rename("median_assets_$m")

dq_flag
False    2220.3
True       48.8
Name: median_assets_$m, dtype: float64

In [10]:
# Are out-of-range returns linked to funds that were winding up?
oor = df[checks["return_out_of_range"]]
print("Out-of-range returns:", len(oor), "| of which have a wind-up date:", oor["wind_up_date"].notna().sum())
oor[show].sort_values("return_1y").head(8)

Out-of-range returns: 13 | of which have a wind-up date: 0


,year,fund_name,fund_type,total_assets_k,member_accounts,return_1y,operating_expense_ratio,duration_months,wind_up_date
1190,2016,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,6809.0,6567.0,-0.856,0.876,12,NaT
1595,2014,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,1466.0,NaN,-0.844,0.864,12,NaT
1384,2015,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,2270.0,6575.0,-0.797,0.840,12,NaT
91,2024,CommInsure Corporate Insurance Superannuation ...,Retail,999.0,0.0,-0.732,0.000,12,NaT
1006,2017,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,2727.0,6624.0,-0.721,0.742,12,NaT
398,2021,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,2737.0,6871.0,-0.705,0.899,12,NaT
180,2023,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,3449.0,6901.0,-0.614,0.822,12,NaT
528,2020,Crown Employees (NSW Fire Brigades Firefightin...,Public Sector,4847.0,6828.0,-0.585,0.764,12,NaT


**What I found after checking the rows above:**
- `any_masked_field` / `missing_assets`: **expected**. APRA masks values to protect privacy, mostly in small funds. I didn't fill these in. I left them out of totals and noted it.
- `return_out_of_range` and `expense_ratio_high`: mostly **tiny funds**. Most rows come from one small NSW public sector defined-benefit scheme with only a few $ million of assets. Its expenses and benefit payments are large compared with its assets, so the ratios blow out (e.g. a -86% "return"). It's real data, but not comparable, so I kept the rows and left them out of return and fee comparisons.
- `non_12_month_duration`: **expected**. These funds started, closed or changed balance date. APRA says their returns shouldn't be compared with 12-month funds.
- `avg_balance_mismatch`: needs a closer look. Usually the two fields are measured at different points or cover different sets of accounts.
- `negative_assets`, `members_but_no_assets`, `missing_fund_type`: 0 rows, so the data passed. A passed check is still worth reporting.
- `dup_abn_period`: **real issue**, see the next section.

## 6b. The duplicate check found something real
Some ABN + period pairs appear twice. Look at them:

In [11]:
dups = df[checks["dup_abn_period"]].sort_values(["abn", "year"])
dups[["year", "fund_name", "abn", "balance_date_group", "balance_date", "duration_months", "total_assets_k"]]

,year,fund_name,abn,balance_date_group,balance_date,duration_months,total_assets_k
1672,2014,National Mutual Pro-Super Fund,46208334003,30 June,2014-06-30,9,NaN
1926,2014,National Mutual Pro-Super Fund,46208334003,Other,2013-09-30,12,NaN
1673,2014,National Mutual Retirement Fund,76746741299,30 June,2014-06-30,9,2861366.0
1927,2014,National Mutual Retirement Fund,76746741299,Other,2013-09-30,12,2971690.0
1548,2014,AMP Super Fund,78421957449,30 June,2014-06-30,9,3978688.0
1934,2014,Super Directions Fund,78421957449,Other,2013-09-30,12,4008391.0
1175,2016,CBH Superannuation Fund,84433159328,30 June,2016-06-30,8,NaN
1866,2016,CBH Superannuation Fund,84433159328,Other,2015-10-31,12,NaN


These funds **changed their balance date** during the year. APRA reports them twice: once in Table 2a
(a short, e.g. 9-month, return to 30 June) and once in Table 2b (12 months to the new date). One ABN even appears under
two names (AMP Super Fund / Super Directions Fund), which is a rename.

If we sum both rows, the fund's assets are **counted twice**. Rule: keep one row per ABN and period, the one with the
**latest balance date**, because it holds the most recent position. Flag it rather than delete it, so it stays traceable.

In [12]:
df["balance_date"] = pd.to_datetime(df["balance_date"], errors="coerce")
latest_first = df.sort_values("balance_date", ascending=False)
df["is_primary_record"] = ~latest_first.duplicated(subset=["abn", "period"], keep="first").reindex(df.index)
print("Rows excluded from totals as duplicates:", (~df["is_primary_record"]).sum())

Rows excluded from totals as duplicates: 4


## 7. numpy: calculations and outliers
### Assets per member (vectorised, no loop)

In [13]:
df["assets_per_member"] = np.where(df["member_accounts"] > 0,
                                   df["total_assets_k"] * 1000 / df["member_accounts"],
                                   np.nan)
df["total_expense_ratio"] = df["operating_expense_ratio"] + df["investment_expenses_ratio"]

### IQR rule on assets per member
Fund size and balance per member are very skewed (a few huge funds). The IQR rule uses quartiles, so the giants don't distort it.

In [14]:
x = df["assets_per_member"].dropna().to_numpy()
q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["apm_outlier"] = (df["assets_per_member"] < low) | (df["assets_per_member"] > high)
print(f"Q1 ${q1:,.0f}  Q3 ${q3:,.0f}  IQR ${iqr:,.0f}  -> outside ${max(low,0):,.0f} to ${high:,.0f}")
print("IQR outliers:", df["apm_outlier"].sum())

Q1 $43,049  Q3 $201,673  IQR $158,624  -> outside $0 to $439,610
IQR outliers: 63


### z-score on total expense ratio (for comparison)

In [15]:
fees = df["total_expense_ratio"].to_numpy(dtype=float)
z = (fees - np.nanmean(fees)) / np.nanstd(fees)
df["fee_z"] = z
df["fee_outlier"] = np.abs(z) > 3
print("z-score fee outliers:", int(np.nansum(df["fee_outlier"])))

# Same rule on assets per member, to show why z-scores fail on skewed data
apm = df["assets_per_member"].to_numpy(dtype=float)
z_apm = (apm - np.nanmean(apm)) / np.nanstd(apm)
print("z-score outliers on assets per member:", int(np.nansum(np.abs(z_apm) > 3)),
      "vs IQR:", int(df["apm_outlier"].sum()))
print("skew of assets per member:", round(pd.Series(apm).skew(), 2))

z-score fee outliers: 11
z-score outliers on assets per member: 29 vs IQR: 63
skew of assets per member: 2.6


**Why IQR over z-score here:** a z-score uses the mean and standard deviation, and extreme values inflate both.
On heavily skewed data, the outliers hide themselves by stretching the standard deviation. Quartiles don't move much
when a few huge values are added, so IQR still catches them.

## 8. Reconcile to the official industry totals
Sum the fund-level total assets by fund type and compare with the Annual Superannuation Bulletin.

In [16]:
bul = pd.read_excel(BULLETIN_FILE, sheet_name="Tab_RSE2_Data")
bul = bul[bul["LINE_ITEM"] == "Total assets at end of the financial year"]
type_map = {"Industry funds": "Industry", "Retail funds": "Retail",
            "Public sector funds": "Public Sector", "Corporate funds": "Corporate"}
year_cols = [c for c in bul.columns if str(c).startswith("Jun ")]
official = (bul[bul["FUNDTYPE_ALL"].isin(type_map)]
            .melt(id_vars="FUNDTYPE_ALL", value_vars=year_cols, var_name="label", value_name="official_assets_m")
            .assign(fund_type_group=lambda d: d["FUNDTYPE_ALL"].map(type_map),
                    year=lambda d: d["label"].str[-4:].astype(int))
            [["year", "fund_type_group", "official_assets_m"]])

mine = (df[df["is_primary_record"]].groupby(["year", "fund_type_group"], as_index=False)
          .agg(my_assets_m=("total_assets_k", lambda s: s.sum() / 1000),
               funds=("fund_name", "count"),
               funds_missing_assets=("total_assets_k", lambda s: s.isna().sum())))

rec = mine.merge(official, on=["year", "fund_type_group"], how="inner")
rec["diff_m"] = rec["my_assets_m"] - rec["official_assets_m"]
rec["diff_pct"] = (rec["diff_m"] / rec["official_assets_m"] * 100).round(2)
rec[rec["year"] == 2025].round(0)

,year,fund_type_group,my_assets_m,funds,funds_missing_assets,official_assets_m,diff_m,diff_pct
40,2025,Corporate,36484.0,2,0,36484.0,0.0,0.0
41,2025,Industry,1566056.0,19,0,1566056.0,-0.0,-0.0
42,2025,Public Sector,595201.0,10,0,773051.0,-177850.0,-23.0
43,2025,Retail,847917.0,46,3,847932.0,-15.0,-0.0


In [17]:
# Does Table 2b matter? Reconcile 2025 using only Table 2a
only_2a = (df[(df["balance_date_group"] == "30 June") & df["is_primary_record"]].groupby(["year", "fund_type_group"])["total_assets_k"].sum() / 1000)
rec_2025 = rec[rec["year"] == 2025].set_index("fund_type_group")
pd.DataFrame({"official_m": rec_2025["official_assets_m"],
              "2a_only_diff_pct": ((only_2a.loc[2025] - rec_2025["official_assets_m"]) / rec_2025["official_assets_m"] * 100).round(2),
              "2a_plus_2b_diff_pct": rec_2025["diff_pct"]})

,official_m,2a_only_diff_pct,2a_plus_2b_diff_pct
fund_type_group,,,
Corporate,36484.0,-20.51,0.00
Industry,1566056.0,-0.00,-0.00
Public Sector,773051.0,-23.01,-23.01
Retail,847932.0,-0.01,-0.00


In [18]:
# Gap over time, per fund type
rec.pivot(index="year", columns="fund_type_group", values="diff_pct")

fund_type_group,Corporate,Industry,Public Sector,Retail
year,,,,
2015,-0.95,0.00,-38.37,-0.24
2016,-2.09,0.00,-37.12,-0.25
2017,-0.34,-0.00,-24.17,-0.49
2018,-0.37,-0.00,-23.53,-0.18
2019,-0.28,-0.00,-22.20,-0.07
2020,-0.15,0.00,-21.69,-0.06
2021,-0.23,-0.00,-21.53,-0.04
2022,-0.80,-0.00,-24.94,-0.30
2023,-0.00,-0.01,-24.02,-0.01


**Findings (2025):**
- **Industry:** matches the bulletin exactly.
- **Corporate:** about 20% short with Table 2a alone, and matches once Table 2b (non-30-June funds) is added. The data was split across two sheets.
- **Retail:** within about 0.002%. The small gap most likely comes from the 3 funds whose assets are masked (`*`).
- **Public Sector:** about 23% short in 2025. The gap is there every year: about 37% in 2015-2016 and 21-25% since 2017. APRA's bulletin notes say the fund-level report *excludes exempt public sector superannuation schemes* (state government schemes that report to APRA voluntarily), while the bulletin includes them. So the gap is a **coverage difference, not a data error**.

## 9. Save clean output (used by Power BI in Project 2)

In [19]:
keep = ["period", "year", "fund_name", "abn", "fund_type", "fund_type_group", "regulatory_class",
        "membership_base", "licensee", "rse_licensee_profit_status", "balance_date_group", "duration_months",
        "wind_up_date", "total_assets_k", "member_accounts", "members_benefits_k", "avg_balance_k",
        "assets_per_member", "return_1y", "return_5y", "return_10y", "operating_expense_ratio",
        "investment_expenses_ratio", "total_expense_ratio", "net_outflow_ratio",
        "masked_field_count", "dq_issue_count", "dq_flag", "is_primary_record", "apm_outlier", "fee_z", "fee_outlier"]
keep = [c for c in keep if c in df.columns]
clean = pd.concat([df[keep], checks.add_prefix("chk_")], axis=1)
clean.to_csv("../data/processed/fund_level_clean.csv", index=False)
summary.to_csv("../data/processed/dq_summary.csv")
rec.to_csv("../data/processed/reconciliation.csv", index=False)
print(clean.shape, "saved")

(1939, 42) saved
